In [8]:
import os
import gc
import json
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.utils import resample
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.metrics import Recall, Precision
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import classification_report, confusion_matrix, precision_score
# --- CONFIGURAÇÕES ---
PASTAS_DATASET = [
    'imagens_acrais_benignas',
    'imagens_acrais_maligno'
]

CAMINHO_MODELOS = 'Modelos'



In [5]:
# Definir Presets
TAMANHO_IMAGEM = (224, 224)

# CARREGAR E COMBINAR METADADOS ---
dfs = []
for pasta in PASTAS_DATASET:
    caminho_csv = os.path.join(pasta, 'metadata.csv')
    if os.path.exists(caminho_csv):
        df_temp = pd.read_csv(caminho_csv)
        # Cria uma nova coluna com o caminho completo da imagem para facilitar
        df_temp['caminho_imagem'] = df_temp['isic_id'].apply(lambda x: os.path.join(pasta, f"{x}.jpg"))
        dfs.append(df_temp)
    else:
        print(f"Aviso: {caminho_csv} não encontrado.")

# Junta os CSVs das duas pastas em um só DataFrame
df_completo = pd.concat(dfs, ignore_index=True)

# FILTRAR DATASET (Usando todas as imagens disponíveis) ---
## 1. Primeiro, removemos duplicatas baseadas no ID único e verificamos existência física
df_unico = df_completo.drop_duplicates(subset='isic_id')
df_existente = df_unico[df_unico['caminho_imagem'].apply(os.path.exists)].copy()

## 2. Separar as classes sem limite de amostras
df_benigno = df_existente[df_existente['diagnosis_1'] == 'Benign']
df_maligno = df_existente[df_existente['diagnosis_1'] == 'Malignant']

## 3. Combinar e embaralhar o dataset final com tudo que foi encontrado
df_filtrado = pd.concat([df_benigno, df_maligno]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset total criado com {len(df_filtrado)} imagens (Benignas: {len(df_benigno)}, Malignas: {len(df_maligno)}).")

Dataset total criado com 2059 imagens (Benignas: 1557, Malignas: 502).


In [6]:
print("=== DADOS: VALIDAÇÃO (20% do df_filtrado) + TREINO EQUILIBRADO (oversampling) ===")

TAMANHO_B0 = (224, 224)
BATCH_SIZE = 32

# Validação: 20% do df_filtrado, sem augmentation (métricas mais estáveis)
datagen_val = ImageDataGenerator(validation_split=0.2)
val_gen = datagen_val.flow_from_dataframe(
    dataframe=df_filtrado,
    x_col="caminho_imagem",
    y_col="diagnosis_1",
    target_size=TAMANHO_B0,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False,
)

# Oversampling da classe minoritária até igualar a majoritária
df_malignant = df_filtrado[df_filtrado["diagnosis_1"] == "Malignant"]
df_benign = df_filtrado[df_filtrado["diagnosis_1"] == "Benign"]
maior_classe = max(len(df_malignant), len(df_benign))
df_benign_upsampled = resample(
    df_benign, replace=True, n_samples=maior_classe, random_state=42
)
df_malignant_upsampled = resample(
    df_malignant, replace=True, n_samples=maior_classe, random_state=42
)
df_equilibrado = pd.concat([df_malignant_upsampled, df_benign_upsampled]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f"Treino equilibrado: {len(df_equilibrado)} imagens")
print(df_equilibrado["diagnosis_1"].value_counts())

datagen_balanced = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=45,       # Rotação padrão (o artigo não especifica 360, então usamos um valor moderado comum)
    width_shift_range=0.1,   # Translações horizontais
    height_shift_range=0.1,  # Translações verticais
    horizontal_flip=True,    # Flips aleatórios
    vertical_flip=True       # Flips aleatórios
    # Sem zoom, sem alteração de cor, sem shear.
)

train_gen_balanced = datagen_balanced.flow_from_dataframe(
    dataframe=df_equilibrado,
    x_col="caminho_imagem",
    y_col="diagnosis_1",
    target_size=TAMANHO_B0,
    batch_size=BATCH_SIZE,
    class_mode="binary",
)

=== DADOS: VALIDAÇÃO (20% do df_filtrado) + TREINO EQUILIBRADO (oversampling) ===
Found 411 validated image filenames belonging to 2 classes.
Treino equilibrado: 3114 imagens
diagnosis_1
Malignant    1557
Benign       1557
Name: count, dtype: int64
Found 3114 validated image filenames belonging to 2 classes.


In [ ]:
# 1. Definindo a grade de testes (3 LRs x 2 Dropouts = 6 Testes)
# learning_rates = [1e-3, 1e-4, 1e-5] 
learning_rates = [1e-4, 1e-5] 
dropouts = [0.4, 0.5]               

# Dicionário para salvar os históricos na memória (útil para a sua próxima célula)
historicos_grid = {}

print("=== INICIANDO GRID SEARCH (6 TESTES) ===")

# O Looping Duplo
for lr in learning_rates:
    for do in dropouts:
        
        # Nome formatado para o teste atual (Ex: LR1e-04_DO0.4)
        nome_teste = f"{CAMINHO_MODELOS}/LR{lr}_DO{do}"
        nome_arquivo = f"{CAMINHO_MODELOS}/treino_{nome_teste}.keras"
        
        print(f"\n" + "="*60)
        print(f"INICIANDO TESTE: {nome_teste}")
        print(f"Taxa de Aprendizado: {lr} | Dropout: {do}")
        print("="*60)
        
        # === PASSO CRÍTICO: LIMPAR MEMÓRIA DA GPU ===
        # Se não fizer isso, a GPU vai travar no 3º ou 4º teste.
        tf.keras.backend.clear_session()
        gc.collect()
        
        # === MONTANDO A REDE DO ZERO PARA ESTE TESTE ===
        base_model = EfficientNetB0(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3)
        )
        base_model.trainable = True # Full Fine-Tuning ativado
        
        x = base_model.output
        x = GlobalAveragePooling2D()(x)
        x = Dropout(do)(x) # Usando o valor de Dropout do Loop
        saida = Dense(1, activation="sigmoid")(x)
        modelo_cnn = Model(inputs=base_model.input, outputs=saida)
        
        # === COMPILANDO ===
        modelo_cnn.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=lr), # Usando o LR do Loop
            loss="binary_crossentropy",
            metrics=["accuracy", tf.keras.metrics.Recall(name="recall")],
        )
        
        # === CALLBACKS DINÂMICOS ===
        callbacks_lista = [
            EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
            ModelCheckpoint(
                filepath=nome_arquivo, # Salva o arquivo com o nome customizado!
                monitor='val_recall',                 
                mode='max',                           
                save_best_only=True,                  
                verbose=1
            ),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=0)
        ]
        
        # === TREINAMENTO ===
        print(f"Treinando modelo {nome_teste}...")
        history = modelo_cnn.fit(
            train_gen_balanced,
            validation_data=val_gen,
            epochs=20, # Reduzi levemente as épocas máximas já que são 6 testes, o EarlyStop para antes se precisar
            callbacks=callbacks_lista,
            verbose=1 # Mantenha 1 para acompanhar o progresso, ou mude para 0 se quiser o terminal limpo
        )
        
        # Salva o histórico de métricas no dicionário
        historicos_grid[nome_teste] = history.history

print("\n✅ GRID SEARCH CONCLUÍDO COM SUCESSO!")
print("Os 6 arquivos .keras foram salvos no seu diretório.")

=== INICIANDO GRID SEARCH (6 TESTES) ===

INICIANDO TESTE: Modelos/LR0.001_DO0.4
Taxa de Aprendizado: 0.001 | Dropout: 0.4
Treinando modelo Modelos/LR0.001_DO0.4...
Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8727 - loss: 0.2952 - recall: 0.9000
Epoch 1: val_recall improved from None to 0.66327, saving model to Modelos/treino_Modelos/LR0.001_DO0.4.keras

Epoch 1: finished saving model to Modelos/treino_Modelos/LR0.001_DO0.4.keras
98/98 ━━━━━━━━━━━━━━━━━━━━ 478s 5s/step - accuracy: 0.9146 - loss: 0.2142 - recall: 0.9281 - val_accuracy: 0.9100 - val_loss: 0.3471 - val_recall: 0.6633 - learning_rate: 0.0010
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.9482 - loss: 0.1349 - recall: 0.9614
Epoch 2: val_recall improved from 0.66327 to 0.67347, saving model to Modelos/treino_Modelos/LR0.001_DO0.4.keras

Epoch 2: finished saving model to Modelos/treino_Modelos/LR0.001_DO0.4.keras
98/98 ━━━━━━━━━━━━━━━━━━━━ 460s 5s/step - accuracy: 0.9512 - loss: 0.1314 - re

In [ ]:
import os
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import gc

learning_rates = [1e-3, 1e-4, 1e-5] 
dropouts = [0.4, 0.5]

print("=== AVALIAÇÃO DOS MODELOS (GRID SEARCH) ===")

y_true = val_gen.classes
class_labels = list(val_gen.class_indices.keys())

# Define a pasta onde você salvou (ajuste se for diferente)
pasta_modelos = "Modelos/treino_Modelos" # Ou "Modelos_TCC", dependendo de como você nomeou

for lr in learning_rates:
    for do in dropouts:
        nome_teste = f"LR{lr}_DO{do}"
        nome_arquivo = f"{nome_teste}.keras"
        
        # O os.path.join coloca a barra '/' automaticamente e do jeito certo
        caminho_completo = os.path.join(pasta_modelos, nome_arquivo)
        
        print(f"\n" + "="*60)
        print(f"📊 AVALIANDO MODELO: {nome_teste}")
        print("="*60)
        
        try:
            # 1. Carrega o modelo treinado
            modelo_salvo = tf.keras.models.load_model(caminho_completo)
            
            # 2. Resetar o gerador de validação
            val_gen.reset()
            
            # 3. Fazer predições
            y_prob = modelo_salvo.predict(val_gen, verbose=0).flatten()
            y_pred = (y_prob > 0.5).astype(int)
            
            # 4. Relatório
            print(f"\nRelatório de Classificação ({nome_teste}):")
            print(classification_report(y_true, y_pred, target_names=class_labels, digits=4, zero_division=0))
            
            # 5. Matriz de Confusão
            cm = confusion_matrix(y_true, y_pred)
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
            plt.title(f"Matriz de Confusão — {nome_teste}")
            plt.ylabel("Real")
            plt.xlabel("Previsto")
            plt.tight_layout()
            plt.show()
            
            # 6. Limpar memória
            del modelo_salvo
            tf.keras.backend.clear_session()
            gc.collect()
            
        except (FileNotFoundError, ValueError): # Agora ele captura o ValueError do Keras 3!
            print(f"⚠️ Arquivo {caminho_completo} não encontrado. Ele pode não ter salvo corretamente ou o treino foi interrompido.")

=== AVALIAÇÃO DOS MODELOS (GRID SEARCH) ===

📊 AVALIANDO MODELO: LR0.0001_DO0.4
⚠️ Arquivo Modelos/treino_Modelos/LR0.0001_DO0.4.keras não encontrado. Ele pode não ter salvo corretamente ou o treino foi interrompido.

📊 AVALIANDO MODELO: LR0.0001_DO0.5
⚠️ Arquivo Modelos/treino_Modelos/LR0.0001_DO0.5.keras não encontrado. Ele pode não ter salvo corretamente ou o treino foi interrompido.

📊 AVALIANDO MODELO: LR1e-05_DO0.4
⚠️ Arquivo Modelos/treino_Modelos/LR1e-05_DO0.4.keras não encontrado. Ele pode não ter salvo corretamente ou o treino foi interrompido.

📊 AVALIANDO MODELO: LR1e-05_DO0.5
⚠️ Arquivo Modelos/treino_Modelos/LR1e-05_DO0.5.keras não encontrado. Ele pode não ter salvo corretamente ou o treino foi interrompido.
